# 직접 해 보기 2 — 파이썬 코드로 따라 해 보기

AI가 내놓은 결과만 보고 그것이 어떻게 나왔는지 모르면, 그 결과를 믿을 근거도 없다. AI도 실수를 하므로 결과는 점검해야 하고, 점검하려면 결과를 만든 파이썬 코드를 읽을 줄 알아야 한다.

이 노트북은 이 장에서 다룬 계산을 파이썬으로 직접 해 본다. 보고서의 세 비교(실험, 일반 인구와의 단순 비교, 조건을 맞춘 비교)를 다시 추정하고, 본문의 표를 차례로 만든다. 새로 배울 함수는 거의 없다. 이 장의 계산은 모두 다중회귀분석 장에서 배운 `smf.ols` 하나로 한다. 달라지는 것은 누구와 비교하느냐(어떤 자료를 합치느냐)와 회귀식에 무엇을 넣느냐뿐이다.

1. 데이터 읽기 — 원 사이트에서 네 파일을 읽고, 한 CSV로 저장한다
2. 세 그룹 살펴보기 — 참가자, 실험 대조군, 일반 인구의 특성(표 3-1)
3. 실험 — 평균의 차이와 회귀식, 변수를 하나씩 더할 때(표 2-2)
4. 일반 인구와의 단순 비교 — 표 3-2
5. 조건을 하나씩 맞추는 계단 — 표 4-1과 보고서의 그림
6. 비교 상대와 넣는 방식을 바꾸면 — 닮은 사람끼리, 곡선으로 넣기, PSID
7. 처치 뒤의 변수 — 표 5-1
8. 늘어난 소득으로 비교하기 — 표 6-1과 대역
9. 더 해 보기: 랄론드의 원래 표본 — 본문에서 다루지 않은 탐색

## 준비 — 패키지와 글꼴

`import`로 쓸 패키지를 불러온다. `as` 뒤의 이름은 줄임말이고, 이후 코드에서는 `pd`, `smf`처럼 줄임말로 부른다. 이 장에서 쓰는 패키지는 모두 Anaconda와 Colab에 기본으로 들어 있다.

matplotlib는 기본 글꼴에 한글이 없어서 그림의 한글이 네모로 깨진다. 그래서 한글 글꼴을 지정한다. 윈도우는 맑은 고딕(`Malgun Gothic`), 맥은 애플고딕(`AppleGothic`)을 쓴다. Colab에는 한글 글꼴이 없으므로 새 셀에서 `!apt-get -qq install fonts-nanum`을 실행하고, 런타임을 다시 시작한 뒤 `NanumGothic`을 지정한다.

In [ ]:
import pandas as pd                      # 표(데이터프레임) 다루기
import statsmodels.formula.api as smf    # 회귀분석
import matplotlib.pyplot as plt          # 그림

# 그림의 한글 글꼴
# 맥은 'AppleGothic', Colab은 'NanumGothic'(설치 필요)
plt.rcParams['font.family'] = 'Malgun Gothic'   # 윈도우
plt.rcParams['axes.unicode_minus'] = False      # 음수 부호(-) 깨짐 방지

## 1. 데이터 읽기

이 장의 자료는 데헤이아(Rajeev Dehejia)와 와바(Sadek Wahba)가 정리한 랄론드(LaLonde)의 NSW 자료다(1절의 "데이터"). 재배포 조건이 밝혀져 있지 않아 책 사이트에 싣지 않았다. 대신 데헤이아의 누리집(`https://users.nber.org/~rdehejia/data/`)에 올라 있는 텍스트 파일을 코드에서 바로 읽는다. 인터넷에 연결돼 있어야 한다.

이 장에서 쓰는 파일은 넷이다. 파일 하나가 한 그룹이다.

| 파일 | 그룹 | 사람 수 |
|---|---|---|
| `nswre74_treated.txt` | 프로그램 참가자 | 185 |
| `nswre74_control.txt` | 추첨에서 떨어진 실험 대조군 | 260 |
| `cps_controls.txt` | 일반 인구 표본(CPS) | 15,992 |
| `psid_controls.txt` | 또 하나의 일반 인구 표본(PSID) | 2,490 |

파일에는 머리글이 없고 숫자만 빈칸으로 나뉘어 있다. 그래서 `sep=r'\s+'`(빈칸 하나 이상으로 나눈다)와 `header=None`(머리글이 없다)을 주고, 열 이름은 `names=`로 직접 붙인다. 열 이름은 원자료의 설명서에 적힌 이름 그대로다.

| 변수 | 뜻 |
|---|---|
| `treat` | 프로그램 참가자이면 1, 아니면 0(본문의 $\textit{treat}$) |
| `age`, `educ` | 나이, 학력(학교에 다닌 햇수) |
| `black`, `hisp` | 흑인이면 1, 히스패닉이면 1 |
| `married` | 결혼했으면 1 |
| `nodegree` | 고등학교를 마치지 않았으면 1(고교 미졸업) |
| `re74`, `re75` | 1974년, 1975년 소득(참가 전, 1982년 달러) |
| `re78` | 1978년 소득(결과, 1982년 달러) |

읽은 뒤에는 보고서에 쓰는 세 그룹(참가자, 실험 대조군, CPS)에 그룹 이름을 `group` 열로 붙여 한 표로 합치고, 직접 해 보기 1에서 AI에게 보여 줄 CSV(`nsw-lalonde.csv`)로 저장한다. 이 CSV는 이 노트북과 같은 폴더에 생긴다.

In [ ]:
URL = 'https://users.nber.org/~rdehejia/data/'
COLS = ['treat', 'age', 'educ', 'black', 'hisp', 'married', 'nodegree', 're74', 're75', 're78']

def read(name):
    return pd.read_csv(URL + name + '.txt', sep=r'\s+', header=None, names=COLS)

treated = read('nswre74_treated')    # 프로그램 참가자
control = read('nswre74_control')    # 실험 대조군
cps = read('cps_controls')           # 일반 인구(CPS)
psid = read('psid_controls')         # 일반 인구(PSID)
for name, g in [('참가자', treated), ('실험 대조군', control), ('CPS', cps), ('PSID', psid)]:
    print(name, len(g), '명')

# 직접 해 보기 1에서 쓸 CSV: 보고서의 세 그룹을 합치고 group 열로 구분한다
allg = pd.concat([treated.assign(group='treated'), control.assign(group='control'), cps.assign(group='cps')])
allg.to_csv('nsw-lalonde.csv', index=False, encoding='utf-8-sig')

## 2. 세 그룹 살펴보기

비교에 쓰는 자료는 둘이다. 참가자와 실험 대조군을 합친 실험 자료(`exp`)와, 같은 참가자와 일반 인구를 합친 관찰 자료(`obs`)다. `pd.concat`은 위아래로 이어 붙인다. 두 자료 모두 참가자는 `treat`가 1, 비교 상대는 0이다.

회귀식에 여러 번 넣을 변수 목록도 여기서 문자열로 만들어 둔다. `X`는 나이, 학력, 인종, 결혼, 고교 미졸업이고, `ALL`은 여기에 참가 전 소득 두 해를 더한 것이다. `steps`는 3절과 5절에서 변수를 하나씩 더할 때 쓰는 차례다.

표 3-1은 세 그룹의 평균이다. 0과 1로 된 변수(흑인, 결혼 같은)의 평균은 1인 사람의 비율이다. `(g['re74'] == 0).mean()`은 1974년 소득이 0인 사람의 비율이다. 앞의 두 열은 닮았는데 셋째 열은 딴판이라는 것을 확인한다(3절).

In [ ]:
exp = pd.concat([treated, control])     # 실험: 추첨으로 나뉜 두 그룹
obs = pd.concat([treated, cps])         # 관찰: 같은 참가자 + 일반 인구

def profile(g):
    return pd.Series({'나이': g['age'].mean(), '학력(년)': g['educ'].mean(),
                      '흑인(%)': g['black'].mean() * 100, '히스패닉(%)': g['hisp'].mean() * 100,
                      '결혼(%)': g['married'].mean() * 100, '고교 미졸업(%)': g['nodegree'].mean() * 100,
                      '1974년 소득': g['re74'].mean(), '1975년 소득': g['re75'].mean(),
                      '1974년 무소득(%)': (g['re74'] == 0).mean() * 100, '1978년 소득(결과)': g['re78'].mean()})

table = pd.DataFrame({'참가자': profile(treated), '실험 대조군': profile(control), '일반 인구(CPS)': profile(cps)})
print(table.round(1))

X = ' + age + educ + black + hisp + married + nodegree'
ALL = X + ' + re75 + re74'
steps = [('없음 (단순 비교)', ''), ('나이, 학력', ' + age + educ'), ('+ 인종, 결혼, 고교 미졸업', X),
         ('+ 1975년 소득', X + ' + re75'), ('+ 1974년 소득', ALL)]

## 3. 실험 — 평균의 차이와 회귀식

추첨으로 나눈 실험에서 프로그램의 효과는 두 그룹의 1978년 평균 소득의 차이다(2절). 이 차이는 `re78`을 `treat` 하나로 회귀한 계수와 같다. 0과 1로 된 변수 하나만 넣은 회귀식의 계수는 1인 그룹과 0인 그룹의 평균 차이이기 때문이다(다중회귀분석 장 B3의 더미 변수).

`m.params['treat']`는 계수, `m.bse['treat']`는 표준오차, `m.conf_int().loc['treat']`는 95% 신뢰구간, `m.pvalues['treat']`는 $p$값이다. 보고서 표의 첫 행과 같아야 한다.

표 2-2는 같은 회귀식에 참가 전 변수를 하나씩 더한 것이다. 추첨으로 나눈 두 그룹은 이미 닮았으니 계수가 크게 바뀌지 않는다. 변수 목록은 2절에서 만든 `steps`를 쓴다.

In [ ]:
diff = treated['re78'].mean() - control['re78'].mean()
print('평균의 차이:', round(diff))

m = smf.ols('re78 ~ treat', exp).fit()
lo, hi = m.conf_int().loc['treat']
print(f"회귀식의 계수: {m.params['treat']:,.0f}  표준오차 {m.bse['treat']:,.0f}  "
      f"t {m.tvalues['treat']:.1f}  95% 신뢰구간 {lo:,.0f} ~ {hi:,.0f}  p {m.pvalues['treat']:.3f}  n {int(m.nobs)}")

for name, rhs in steps:
    r = smf.ols('re78 ~ treat' + rhs, exp).fit()
    print(f"표 2-2  {name:16s} {r.params['treat']:7,.0f}  ({r.bse['treat']:,.0f})  p {r.pvalues['treat']:.3f}")

## 4. 일반 인구와의 단순 비교

같은 회귀식을 관찰 자료에 적용한다. 비교 상대만 실험 대조군에서 일반 인구로 바꾼 것이다. 계수가 마이너스 8,498달러로, 실험의 답과 부호부터 반대가 된다(3절의 표 3-2). 일반 인구는 참가 전부터 소득이 훨씬 높던 사람들이기 때문이다.

In [ ]:
m = smf.ols('re78 ~ treat', obs).fit()
lo, hi = m.conf_int().loc['treat']
print(f"계수 {m.params['treat']:,.0f}  표준오차 {m.bse['treat']:,.0f}  t {m.tvalues['treat']:.1f}  "
      f"95% 신뢰구간 {lo:,.0f} ~ {hi:,.0f}  n {int(m.nobs)}  R2 {m.rsquared:.3f}")

## 5. 조건을 하나씩 맞추는 계단

관찰 자료의 회귀식에 3절과 같은 변수를 하나씩 더한다(4절의 표 4-1). 결과를 `stair` 표에 모아 두고, 보고서 그림의 오른쪽 패널처럼 점을 이어 그린다. 1975년 소득을 넣는 순간 계수가 플러스로 뛰어오르는 것을 확인한다.

4절은 나이, 학력 같은 다른 변수는 하나도 넣지 않고 참가 전 소득 두 개만 넣어도 계수가 169달러가 된다고 했다. 그것도 계산해 본다.

그림의 주황 점선은 실험의 답이고, 옅은 띠는 그 95% 신뢰구간이다. `axhspan`은 가로 띠를, `axhline`은 가로선을 그린다.

In [ ]:
rows = []
for name, rhs in steps:
    r = smf.ols('re78 ~ treat' + rhs, obs).fit()
    rows.append({'함께 넣은 변수': name, '계수': r.params['treat'], '표준오차': r.bse['treat'],
                 't': r.tvalues['treat'], 'p': r.pvalues['treat'], 'R2': r.rsquared})
stair = pd.DataFrame(rows)
print(stair.round({'계수': 0, '표준오차': 0, 't': 1, 'p': 3, 'R2': 3}))

two = smf.ols('re78 ~ treat + re74 + re75', obs).fit()
print('\n참가 전 소득 두 개만 넣으면:', round(two.params['treat']))

e = smf.ols('re78 ~ treat', exp).fit()          # 실험의 답
lo, hi = e.conf_int().loc['treat']
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.axhline(0, color='lightgray', lw=0.8)
ax.axhspan(lo, hi, color='tab:orange', alpha=0.12)
ax.axhline(e.params['treat'], color='tab:orange', ls='--', label=f"실험의 답 {e.params['treat']:,.0f}")
ax.plot(range(len(stair)), stair['계수'], marker='o', color='tab:blue')
ax.set_xticks(range(len(stair)))
ax.set_xticklabels(['단순\n비교', '+ 나이,\n학력', '+ 인종, 결혼\n고교 미졸업', '+ 1975년\n소득', '+ 1974년\n소득'], fontsize=8)
ax.set_ylabel('treat의 계수 (달러)')
ax.set_title('일반 인구와 비교하며 조건을 하나씩 맞출 때')
ax.legend()
plt.tight_layout()
plt.show()

## 6. 비교 상대와 넣는 방식을 바꾸면

4절의 "닮은 사람끼리"는 일반 인구 가운데 참가 전 두 해의 소득이 모두 5,000달러 미만인 사람만 남겨 비교 상대로 삼았다. `cps[(조건1) & (조건2)]`는 두 조건을 모두 만족하는 행만 남긴다. `&`는 "그리고"다.

4절의 "아직 모자라다"는 같은 변수를 넣는 방식만 바꿔도 숫자가 움직인다고 했다. `CURVE`는 나이와 학력의 제곱항(`I(age**2)`)과, 참가 전 소득이 0이었는지를 나타내는 더미(`I(re74 == 0)`)를 더한 회귀식이다. 회귀식 안의 `I( )`는 괄호 안의 계산을 그대로 하라는 표시다(다중회귀분석 장 B5의 제곱항).

같은 두 회귀식을 PSID와 비교해서도 계산한다. 실험의 답은 넣는 방식을 바꿔도 크게 움직이지 않지만, 관찰 비교는 비교 상대와 넣는 방식에 따라 239달러에서 1,528달러까지 흩어지는 것을 확인한다.

In [ ]:
low = cps[(cps['re74'] < 5000) & (cps['re75'] < 5000)]     # 닮은 사람: 참가 전 두 해 소득이 모두 5,000달러 미만
obl = pd.concat([treated, low])
print('닮은 사람', len(low), '명, 나이', round(low['age'].mean(), 1), ', 결혼', round(low['married'].mean() * 100), '%')
print('  단순 비교', round(smf.ols('re78 ~ treat', obl).fit().params['treat']),
      ' 변수를 모두', round(smf.ols('re78 ~ treat' + ALL, obl).fit().params['treat']))

CURVE = ALL + ' + I(age**2) + I(educ**2) + I(re74 == 0) + I(re75 == 0)'
obp = pd.concat([treated, psid])
for name, d in [('실험', exp), ('일반 인구(CPS)', obs), ('닮은 사람', obl), ('일반 인구(PSID)', obp)]:
    a = smf.ols('re78 ~ treat' + ALL, d).fit().params['treat']
    b = smf.ols('re78 ~ treat' + CURVE, d).fit().params['treat']
    print(f'{name:12s} 직선으로 {a:7,.0f}   곡선까지 {b:7,.0f}')

## 7. 처치 뒤의 변수

5절의 표 5-1은 실험의 회귀식에 "1978년에 일했나"를 넣었다. 1978년 근로소득이 0보다 컸으면 1인 더미다. `(exp['re78'] > 0).astype(int)`는 참·거짓을 1과 0으로 바꾼다. `exp.assign(...)`은 원래 표는 그대로 두고 열을 하나 더한 새 표를 만든다.

이 변수는 프로그램에 참가한 뒤에 정해진 것이라, 넣으면 프로그램이 사람들을 일하게 만들어 올린 소득이 비교에서 빠진다. $\textit{treat}$의 계수가 1,794달러에서 961달러로 줄고, $R^2$은 크게 오르는 것을 확인한다. `groupby('treat')`로 두 그룹에서 일한 사람의 비율도 센다.

In [ ]:
ew = exp.assign(work78=(exp['re78'] > 0).astype(int))
for rhs in ['', ' + work78']:
    r = smf.ols('re78 ~ treat' + rhs, ew).fit()
    print(f"re78 ~ treat{rhs:9s}  계수 {r.params['treat']:6,.0f}  ({r.bse['treat']:,.0f})  p {r.pvalues['treat']:.3f}  R2 {r.rsquared:.3f}")
rate = ew.groupby('treat')['work78'].mean()
print(f"1978년에 일한 비율: 참가자 {rate[1]:.0%}, 실험 대조군 {rate[0]:.0%}")

## 8. 늘어난 소득으로 비교하기

6절의 표 6-1 (2)열은 종속변수를 1978년 소득 대신 늘어난 소득(`re78 - re75`)으로 바꾸고 `treat` 하나만 넣었다. 비교 상대를 실험 대조군과 일반 인구로 바꿔 가며 계산한다. 회귀식 안에서 `I(re78 - re75)`라고 쓰면 따로 열을 만들지 않아도 된다.

같은 값을 그림 6-2의 대역으로도 계산한다. 참가자의 1975년 평균 소득에 비교 상대가 늘어난 만큼을 더한 것이 "참가하지 않았다면"의 1978년 소득(대역)이고, 실제 1978년 평균 소득과 대역의 차이가 회귀식의 계수와 같다.

마지막으로 처치 전 추세를 본다. 1974년에서 1975년까지 평균 소득이 얼마나 바뀌었는지를 그룹마다 센다(6절의 "나란히 움직였을까").

In [ ]:
for name, d in [('실험 대조군', exp), ('일반 인구(CPS)', obs)]:
    r = smf.ols('I(re78 - re75) ~ treat', d).fit()
    print(f"{name:10s} 늘어난 소득의 비교 {r.params['treat']:6,.0f}  ({r.bse['treat']:,.0f})")

t75, t78 = treated['re75'].mean(), treated['re78'].mean()
for name, g in [('일반 인구만큼', cps), ('실험 대조군만큼', control)]:
    gain = g['re78'].mean() - g['re75'].mean()
    print(f"대역({name} 늘었다면): {t75:,.0f} + {gain:,.0f} = {t75 + gain:,.0f},  실제 {t78:,.0f}와의 차이 {t78 - t75 - gain:,.0f}")

for name, g in [('참가자', treated), ('실험 대조군', control), ('일반 인구(CPS)', cps)]:
    print(f"{name:10s} 1974 → 1975년 평균 소득의 변화 {g['re75'].mean() - g['re74'].mean():6,.0f}")

## 9. 더 해 보기: 랄론드의 원래 표본

본문에서는 다루지 않은 탐색이다. 이 장의 자료는 데헤이아와 와바가 랄론드의 남성 표본에서 1974년 소득이 있는 사람만 남긴 것이다. 랄론드가 1986년 논문에서 쓴 원래 표본(참가자 297명, 대조군 425명)은 같은 누리집의 `nsw_treated.txt`, `nsw_control.txt`에 있다. 이 파일에는 1974년 소득 열이 없어서 열 이름도 하나 적다.

원래 표본의 실험의 답은 이 장의 1,794달러보다 훨씬 작다. 어떤 사람을 표본에 남기느냐만으로도 실험의 답이 달라진다는 것이고, 이 때문에 데헤이아·와바의 표본을 두고 연구자들 사이에 논쟁이 있다. 이 장의 결론(관찰 비교는 비교 상대와 가정에 따라 크게 흔들린다)은 어느 표본에서도 같다.

In [ ]:
C9 = [c for c in COLS if c != 're74']
orig = pd.concat([pd.read_csv(URL + n + '.txt', sep=r'\s+', header=None, names=C9) for n in ['nsw_treated', 'nsw_control']])
r = smf.ols('re78 ~ treat', orig).fit()
print(f"원래 표본 {len(orig)}명 (참가자 {int(orig['treat'].sum())}명): 실험의 답 {r.params['treat']:,.0f}  ({r.bse['treat']:,.0f})")

## 정리

이 노트북의 코드는 이 장에서 본 숫자를 직접 만들어 냈다. 계산은 모두 `smf.ols` 하나였다. 보고서의 세 비교는 3, 4, 5절에서, 본문의 표는 같은 회귀식의 비교 상대를 바꾸거나(6절), 넣는 변수를 바꾸거나(5~7절), 종속변수를 바꾼(8절) 것이다. 같은 계산 도구로 이렇게 다른 답이 나온다는 것이 이 장의 이야기다. 숫자를 효과로 해석할 수 있느냐는 코드가 아니라 비교의 설계가 정한다.

자기 자료로 분석하려면 1절에서 자기 CSV를 `pd.read_csv`로 읽고, 처치 변수를 `treat`, 결과 변수를 `re78` 자리에 바꿔 넣는다. 2절의 `X`와 `steps`에는 자기 자료의 처치 전 변수를 적는다. 처치 뒤에 측정한 변수는 넣지 않는다(7절). 실험 대조군이 없으면 3절과 6~8절의 실험 비교는 건너뛰고, 처치 전후의 결과가 없으면 8절도 건너뛴다. AI가 만든 보고서를 받았다면, 이 코드로 같은 숫자가 나오는지 확인해 보는 것이 가장 확실한 점검이다.